# Lab 2 — Adaptive RAG: Classificador com 3 Caminhos
## MBA em RAG & CAG Aplicados a Direito e Segurança Pública | Aula 10

**Objetivo:** Implementar o Adaptive RAG com LangGraph, roteando queries jurídicas para:
- Caminho 1: Sem retrieval (conhecimento do LLM)
- Caminho 2: Single-step RAG (1 busca)
- Caminho 3: Multi-step RAG (agente completo)

**Duração estimada:** 90 minutos

---

**Referência:** JEONG, Soyeong et al. **Adaptive-RAG: Learning to Adapt Retrieval-Augmented Large Language Models through Question Complexity**. ACL Findings, 2024.

In [ ]:
!pip install -q langgraph langchain langchain-openai langchain-community pydantic

import os
import sqlite3
import time
import pandas as pd
import matplotlib.pyplot as plt
from typing import TypedDict, Literal, Annotated, Sequence
import operator

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, BaseMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from pydantic import BaseModel, Field

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
except:
    pass

DB_PATH = "../datasets/juridico_segpub.db"
print("✅ Ambiente pronto")

In [ ]:
# PASSO 1: Classificador de Complexidade
# O coração do Adaptive RAG — determina qual caminho seguir

class ClassificacaoQuery(BaseModel):
    """Classificação estruturada da complexidade da query."""
    complexidade: Literal["sem_retrieval", "single_step", "multi_step"] = Field(
        description=(
            "sem_retrieval: LLM responde com conhecimento próprio — definições gerais, conceitos básicos;\n"
            "single_step: 1 busca resolve — prazo específico, artigo de lei, resultado de caso;\n"
            "multi_step: múltiplas buscas — comparação, análise de tendência, pesquisa multi-fonte"
        )
    )
    justificativa: str = Field(description="Por que essa classificação? (1-2 frases)")
    confianca: float = Field(description="Confiança de 0.0 a 1.0", ge=0.0, le=1.0)
    subtipo: str = Field(description="Tipo específico da query (ex: 'definição', 'prazo', 'precedente', 'análise')")


CLASSIFIER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """
Você é um classificador de complexidade para um sistema RAG jurídico.
Analise a pergunta e classifique em um dos 3 níveis:

📗 SEM_RETRIEVAL — LLM responde sem buscar:
  • Definições de conceitos jurídicos (o que é habeas corpus?)
  • Explicações de institutos do direito
  • Procedimentos gerais amplamente conhecidos

📘 SINGLE_STEP — Uma busca resolve:
  • Prazo processual específico
  • Artigo ou inciso de lei específico
  • Resultado de um caso judicial específico
  • Quem é o relator de um caso

📕 MULTI_STEP — Múltiplas buscas necessárias:
  • Comparação entre casos ou legislações
  • Análise de tendência jurisprudencial
  • Pesquisa que envolve múltiplas fontes
  • Parecer que requer síntese de informações diversas
"""),
    ("human", "Classifique: {query}")
])

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_classifier = llm.with_structured_output(ClassificacaoQuery)
classifier_chain = CLASSIFIER_PROMPT | llm_classifier

print("✅ Classificador configurado")

# Teste rápido do classificador
queries_rapidas = [
    "O que é o princípio da presunção de inocência?",
    "Qual é o prazo para o recurso de apelação criminal?",
    "Compare a jurisprudência do STF sobre prisão preventiva em 2022 e 2023"
]

if os.getenv("OPENAI_API_KEY"):
    print("\n🧪 TESTE DO CLASSIFICADOR:\n")
    for q in queries_rapidas:
        clf = classifier_chain.invoke({"query": q})
        icon = {"sem_retrieval": "📗", "single_step": "📘", "multi_step": "📕"}.get(clf.complexidade, "❓")
        print(f"{icon} [{clf.complexidade:>15}] {q[:60]}")
        print(f"   Justificativa: {clf.justificativa}")
        print(f"   Confiança: {clf.confianca:.0%} | Subtipo: {clf.subtipo}\n")
else:
    print("ℹ️  Configure OPENAI_API_KEY para testar o classificador")

In [ ]:
# PASSO 2: Ferramentas do Agente (para o caminho multi_step)

@tool
def search_docs(query: str) -> str:
    """Busca jurisprudência e acórdãos no banco local."""
    try:
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        palavra = query.lower().split()[0] if query else ""
        cursor.execute(
            "SELECT numero, tribunal, ementa, resultado FROM acordaos WHERE LOWER(ementa) LIKE ? LIMIT 3",
            (f"%{palavra}%",)
        )
        rows = cursor.fetchall()
        conn.close()
        if not rows:
            return f"Sem resultados para: '{query}'"
        return "\n".join(f"📋 {n} ({t}): {e[:150]} → {r}" for n, t, e, r in rows)
    except Exception as e:
        return f"Banco de dados não encontrado: {e}. Execute o Lab 1 primeiro."

@tool
def query_db(sql_query: str) -> str:
    """Executa SQL em tabelas: acordaos, ocorrencias, legislacao."""
    if not sql_query.strip().upper().startswith("SELECT"):
        return "❌ Apenas SELECT."
    try:
        conn = sqlite3.connect(DB_PATH)
        conn.row_factory = sqlite3.Row
        rows = conn.execute(sql_query).fetchall()
        conn.close()
        if not rows:
            return "Sem resultados."
        cols = list(rows[0].keys())
        return f"{len(rows)} registros | " + " | ".join(
            str(rows[0][c])[:60] for c in cols
        )
    except Exception as e:
        return f"Erro SQL: {e}"

tools_agente = [search_docs, query_db]
llm_agent = llm.bind_tools(tools_agente)
print("✅ Ferramentas para multi_step configuradas")

In [ ]:
# PASSO 3: Estado e Nós do Adaptive RAG

class AdaptiveRAGState(TypedDict):
    query: str
    complexidade: str
    justificativa: str
    confianca: float
    subtipo: str
    contexto: str
    resposta: str
    tool_calls_count: int
    latencia_ms: float


# NÓ 1: Classificador
def nó_classificar(state: AdaptiveRAGState) -> dict:
    """Classifica a complexidade da query."""
    t0 = time.time()
    try:
        clf = classifier_chain.invoke({"query": state["query"]})
        return {
            "complexidade": clf.complexidade,
            "justificativa": clf.justificativa,
            "confianca": clf.confianca,
            "subtipo": clf.subtipo
        }
    except Exception as e:
        # Fallback: single_step se classificação falhar
        return {"complexidade": "single_step", "justificativa": f"Fallback: {e}",
                "confianca": 0.5, "subtipo": "desconhecido"}


# NÓ 2: Sem retrieval
def nó_sem_retrieval(state: AdaptiveRAGState) -> dict:
    """Caminho 1: LLM responde diretamente."""
    t0 = time.time()
    resp = llm.invoke([
        SystemMessage(content="Você é um assistente jurídico especializado. Responda de forma clara e objetiva."),
        HumanMessage(content=state["query"])
    ])
    return {
        "resposta": resp.content,
        "contexto": "[Sem retrieval — resposta do LLM]",
        "tool_calls_count": 0,
        "latencia_ms": (time.time() - t0) * 1000
    }


# NÓ 3: Single-step RAG
def nó_single_step(state: AdaptiveRAGState) -> dict:
    """Caminho 2: Uma busca + geração."""
    t0 = time.time()
    ctx = search_docs.invoke({"query": state["query"]})
    resp = llm.invoke([
        SystemMessage(content="Responda baseado no contexto. Cite as fontes."),
        HumanMessage(content=f"Contexto:\n{ctx}\n\nPergunta: {state['query']}")
    ])
    return {
        "resposta": resp.content,
        "contexto": ctx,
        "tool_calls_count": 1,
        "latencia_ms": (time.time() - t0) * 1000
    }


# NÓ 4: Multi-step (Agente completo)
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]
    iteration_count: int

def _agent_fn(state: AgentState) -> dict:
    it = state.get("iteration_count", 0)
    if it >= 6:
        return {"messages": [AIMessage(content="[Limite atingido]")], "iteration_count": it}
    r = llm_agent.invoke(list(state["messages"]))
    return {"messages": [r], "iteration_count": it + 1}

def _route(state: AgentState) -> str:
    last = state["messages"][-1]
    return "tools" if (hasattr(last, "tool_calls") and last.tool_calls) else "end"

_wf = StateGraph(AgentState)
_wf.add_node("agent", _agent_fn)
_wf.add_node("tools", ToolNode(tools_agente))
_wf.set_entry_point("agent")
_wf.add_conditional_edges("agent", _route, {"tools": "tools", "end": END})
_wf.add_edge("tools", "agent")
_sub_app = _wf.compile()

def nó_multi_step(state: AdaptiveRAGState) -> dict:
    """Caminho 3: Agente ReAct completo."""
    t0 = time.time()
    inputs = {"messages": [HumanMessage(content=state["query"])], "iteration_count": 0}
    ctxs, resp_final, tc_count = [], "", 0
    for event in _sub_app.stream(inputs, {"recursion_limit": 20}):
        for _, out in event.items():
            for msg in out.get("messages", []):
                if hasattr(msg, "tool_calls") and msg.tool_calls:
                    tc_count += len(msg.tool_calls)
                elif isinstance(msg, ToolMessage):
                    ctxs.append(str(msg.content)[:200])
                elif hasattr(msg, "content") and isinstance(msg.content, str) and len(msg.content) > 50:
                    resp_final = msg.content
    return {
        "resposta": resp_final,
        "contexto": "\n---\n".join(ctxs),
        "tool_calls_count": tc_count,
        "latencia_ms": (time.time() - t0) * 1000
    }


# Função de roteamento
def rotear(state: AdaptiveRAGState) -> str:
    return state["complexidade"]


print("✅ Nós do Adaptive RAG configurados")

In [ ]:
# PASSO 4: Montar o Grafo Adaptive RAG

adaptive_wf = StateGraph(AdaptiveRAGState)

adaptive_wf.add_node("classificar", nó_classificar)
adaptive_wf.add_node("sem_retrieval", nó_sem_retrieval)
adaptive_wf.add_node("single_step", nó_single_step)
adaptive_wf.add_node("multi_step", nó_multi_step)

adaptive_wf.set_entry_point("classificar")

adaptive_wf.add_conditional_edges(
    "classificar",
    rotear,
    {
        "sem_retrieval": "sem_retrieval",
        "single_step": "single_step",
        "multi_step": "multi_step"
    }
)

adaptive_wf.add_edge("sem_retrieval", END)
adaptive_wf.add_edge("single_step", END)
adaptive_wf.add_edge("multi_step", END)

adaptive_app = adaptive_wf.compile()

print("✅ Adaptive RAG compilado!")
print("""
🗺️  Fluxo:
[START] → [CLASSIFICAR] → sem_retrieval → [LLM DIRETO]  → [END]
                        → single_step   → [1 BUSCA]     → [END]
                        → multi_step    → [AGENTE RAG]  → [END]
""")

In [ ]:
# PASSO 5: Testar com queries de diferentes complexidades

queries_benchmark = [
    # (query, complexidade_esperada, descricao)
    ("O que é habeas corpus?", "sem_retrieval", "Definição geral"),
    ("Explique o princípio da presunção de inocência", "sem_retrieval", "Conceito jurídico"),
    ("Qual é o prazo para recurso de apelação criminal?", "single_step", "Prazo processual"),
    ("O que diz o Art. 41 do CPP?", "single_step", "Artigo específico"),
    ("Qual o resultado do HC 123456 no STF?", "single_step", "Caso específico"),
    ("Compare os precedentes STF de prisão preventiva em crimes financeiros vs violentos",
     "multi_step", "Análise comparativa"),
    ("Faça um parecer sobre lavagem de dinheiro via cripto considerando lei e jurisprudência atual",
     "multi_step", "Parecer complexo"),
]

def executar_benchmark(queries: list) -> pd.DataFrame:
    """Executa o Adaptive RAG com múltiplas queries e coleta métricas."""
    resultados = []
    
    for query, esperado, desc in queries:
        print(f"\n🔍 Testando: {desc}")
        print(f"   Query: {query[:70]}..." if len(query) > 70 else f"   Query: {query}")
        
        state_inicial = {
            "query": query,
            "complexidade": "",
            "justificativa": "",
            "confianca": 0.0,
            "subtipo": "",
            "contexto": "",
            "resposta": "",
            "tool_calls_count": 0,
            "latencia_ms": 0.0
        }
        
        t0 = time.time()
        resultado = adaptive_app.invoke(state_inicial)
        total_ms = (time.time() - t0) * 1000
        
        acertou = resultado["complexidade"] == esperado
        icon = "✅" if acertou else "❌"
        print(f"   {icon} Classificado: {resultado['complexidade']} (esperado: {esperado})")
        print(f"   ⏱️  Latência: {resultado['latencia_ms']:.0f}ms | Tool calls: {resultado['tool_calls_count']}")
        
        resultados.append({
            "descricao": desc,
            "query": query[:60],
            "esperado": esperado,
            "classificado": resultado["complexidade"],
            "acertou": acertou,
            "confianca": resultado["confianca"],
            "tool_calls": resultado["tool_calls_count"],
            "latencia_ms": resultado["latencia_ms"]
        })
    
    return pd.DataFrame(resultados)


if os.getenv("OPENAI_API_KEY"):
    df_resultados = executar_benchmark(queries_benchmark)
    
    print(f"\n{'='*70}")
    print("📊 RESUMO DO BENCHMARK")
    print(f"{'='*70}")
    acuracia = df_resultados["acertou"].mean()
    print(f"\n✅ Acurácia do classificador: {acuracia:.0%}")
    print(f"⏱️  Latência média por caminho:")
    for caminho in ["sem_retrieval", "single_step", "multi_step"]:
        subset = df_resultados[df_resultados["classificado"] == caminho]
        if len(subset) > 0:
            print(f"   {caminho}: {subset['latencia_ms'].mean():.0f}ms (n={len(subset)})")
else:
    print("ℹ️  Configure OPENAI_API_KEY para executar o benchmark completo")
    
    # Simula resultado para demonstração
    df_resultados = pd.DataFrame([
        {"descricao": "Definição geral", "esperado": "sem_retrieval", "classificado": "sem_retrieval",
         "acertou": True, "confianca": 0.95, "tool_calls": 0, "latencia_ms": 1200},
        {"descricao": "Prazo processual", "esperado": "single_step", "classificado": "single_step",
         "acertou": True, "confianca": 0.88, "tool_calls": 1, "latencia_ms": 3500},
        {"descricao": "Análise comparativa", "esperado": "multi_step", "classificado": "multi_step",
         "acertou": True, "confianca": 0.92, "tool_calls": 4, "latencia_ms": 18000},
    ])
    print("(Usando dados simulados para demonstração)")

print("\n" + df_resultados.to_string(index=False))

In [ ]:
# PASSO 6: Visualização — Distribuição de Rotas e Impacto no Custo

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Análise do Adaptive RAG — Distribuição e Custos", fontsize=14, fontweight="bold")

# Grafico 1: Distribuição das rotas
dist = df_resultados["classificado"].value_counts()
cores = ["#2ecc71", "#3498db", "#e74c3c"]
axes[0].pie(dist.values, labels=dist.index, autopct='%1.0f%%', colors=cores[:len(dist)])
axes[0].set_title("Distribuição de Rotas")

# Gráfico 2: Latência por caminho
if len(df_resultados) > 0 and "latencia_ms" in df_resultados:
    lat_data = df_resultados.groupby("classificado")["latencia_ms"].mean()
    axes[1].bar(lat_data.index, lat_data.values / 1000, color=cores[:len(lat_data)])
    axes[1].set_title("Latência Média por Caminho")
    axes[1].set_ylabel("Segundos")
    axes[1].tick_params(axis='x', rotation=15)

# Gráfico 3: Tool calls por caminho
if "tool_calls" in df_resultados:
    tc_data = df_resultados.groupby("classificado")["tool_calls"].mean()
    axes[2].bar(tc_data.index, tc_data.values, color=cores[:len(tc_data)])
    axes[2].set_title("Média de Tool Calls por Caminho")
    axes[2].set_ylabel("Nº de Tool Calls")
    axes[2].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig("adaptive_rag_analise.png", dpi=100, bbox_inches="tight")
plt.show()
print("✅ Gráfico salvo em: adaptive_rag_analise.png")

# Impacto estimado no custo
print("\n💰 IMPACTO ESTIMADO NO CUSTO (gpt-4o-mini, 1000 queries/dia):")
custo_por_rota = {"sem_retrieval": 0.0002, "single_step": 0.0005, "multi_step": 0.003}
for rota, custo in custo_por_rota.items():
    n = len(df_resultados[df_resultados["classificado"] == rota])
    pct = n / len(df_resultados) if len(df_resultados) > 0 else 0.33
    custo_dia = pct * 1000 * custo
    print(f"  {rota:<18} {pct:.0%} das queries → ${custo_dia:.2f}/dia → ${custo_dia*30:.2f}/mês")

---
## ✏️ Exercício

**Tarefa:** Adicione um 4° caminho ao Adaptive RAG: `"cached"` — quando a mesma query (ou muito similar) já foi respondida antes.

**Dicas:**
1. Adicione um campo `cache: dict` ao `AdaptiveRAGState`
2. Modifique o classificador para detectar queries em cache
3. Crie um nó `nó_cached` que retorna a resposta do cache

**Benefício esperado:** Redução de ~30% no custo em sistemas de produção com queries repetitivas.